In [ ]:
%matplotlib widget

import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
from numpy import pi
import time
import torch.nn as nn
import pypulseq as pp
import torch
from Modules.CoilSens import cyclic_head_coil
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import ChannelOperator, LinearOperator, IdentityOperator
from Modules.NumSolvers import conjugate_gradient
import Modules.ConcatSum
from Modules.Plan import KbNufft, KbNufftAdjoint
from tqdm import tqdm
from Modules.Regularizer import Regularizer
from Modules.fast_iso import isochromat_sim_fast

sys.path.insert(0, "..")
from Modules import default_system, load_brain_phantom

torch.set_default_device("cuda")

In [ ]:
def truefisp_seq_3d(
    FOV: float = 250e-3,
    N_x: int = 128,
    FOV_z: float = 80e-3,
    N_z: int = 16,
    alpha: float = 60.0,
    slab_thickness: float = 64e-3,
    TRF: float = 0.8e-3,
    TBW: float = 2.0,
    BW: float = 890.0,
    pre_dur: float = 0.6e-3,
    TR: float = 3.12e-3,
    N_readout: int = 1800,
    N_spokes: int = 1,
    R=1,
    N_dummy: int = 0,
    TI_start: float = 20e-3,
    TI_inter: float = 0.0,
    TD: float = 2000e-3,
    inv_dur: float = 2e-3,
    kz_order: str = "golden",  # "golden" or "sequential"
    write_seq: bool = False,
    plot: bool = False,
    return_traj: bool = False,
):
    """
    3-D stack-of-stars IR-TrueFISP (bSSFP).
      - in-plane : golden-angle radial readout (as in the 2-D version)
      - through-plane : Cartesian partition encoding (kz) on a slab-selective excitation
    kz_order:
      "golden"     -> angle and kz index both advance every TR (golden angle / golden ratio),
                      so every time point after inversion samples a quasi-uniform (angle, kz) set
      "sequential" -> all N_z partitions are acquired for one angle before moving to the next angle
    """

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / FOV
    deltak_z = 1 / FOV_z
    _gr = seq.grad_raster_time
    GA_DEG = 111.246
    GR = 0.6180339887498949

    # ---- RF -------------------------------------------------------------
    # Non-selective inversion (whole volume, so the slab is inverted regardless of position)
    rf_inv = pp.make_block_pulse(
        flip_angle=np.deg2rad(180),
        duration=inv_dur,
        system=system,
        use="inversion",
    )

    # Slab-selective excitation (TRF / TBW are now actually used)
    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=TRF,
        flip_angle=np.deg2rad(alpha),
        slice_thickness=slab_thickness,
        system=system,
        time_bw_product=TBW,
        return_gz=True,
        use="excitation",
    )

    # ---- Readout --------------------------------------------------------
    _astep = N_x * seq.adc_raster_time
    _k = 1
    while abs(_k * _gr - round(_k * _gr / _astep) * _astep) > 1e-9:
        _k += 1
    readout_flat = int(round((1.0 / BW) / (_k * _gr))) * (_k * _gr)

    gx_base = pp.make_trapezoid(
        channel="x", flat_area=N_x * deltak, flat_time=readout_flat, system=system
    )
    adc = pp.make_adc(
        num_samples=N_x,
        duration=gx_base.flat_time,
        delay=gx_base.rise_time,
        system=system,
    )

    # ---- Pre-phasers / rewinders (shared duration for x, y, z) -----------
    kz_idx_max = N_z // 2
    kz_area_max = max(abs(i - N_z // 2) for i in range(N_z)) * deltak_z
    z_area_max = abs(gz.area) / 2 + kz_area_max
    x_area_max = abs(gx_base.area) / 2

    min_x = pp.calc_duration(pp.make_trapezoid(channel="x", area=x_area_max, system=system))
    min_z = pp.calc_duration(pp.make_trapezoid(channel="z", area=z_area_max, system=system))
    pre_dur = int(np.ceil(max(pre_dur, min_x, min_z) / _gr - 1e-9)) * _gr

    gx_pre_base = pp.make_trapezoid(
        channel="x", area=-gx_base.area / 2, duration=pre_dur, system=system
    )

    # z pre-phaser = slab rephasing + partition encode; z rewinder undoes the partition
    # encode and the second half of the slab rephasing -> net z moment per TR is zero.
    pre_z_cache, rew_z_cache = {}, {}

    def get_gz_pre_rew(iz):
        if iz not in pre_z_cache:
            kz_area = (iz - N_z // 2) * deltak_z
            pre_z_cache[iz] = pp.make_trapezoid(
                channel="z", area=-gz.area / 2 + kz_area, duration=pre_dur, system=system
            )
            rew_z_cache[iz] = pp.make_trapezoid(
                channel="z", area=-gz.area / 2 - kz_area, duration=pre_dur, system=system
            )
        return pre_z_cache[iz], rew_z_cache[iz]

    # ---- Timing ---------------------------------------------------------
    dur_exc = pp.calc_duration(rf, gz)
    dur_ro = pp.calc_duration(gx_base)
    free = TR - dur_exc - 2 * pre_dur - dur_ro
    if free < 0:
        print(
            f"WARNING: requested TR={TR*1e3:.2f} ms not achievable; "
            f"minimum TR = {(TR - free)*1e3:.2f} ms"
        )
    delay_TE = int(np.ceil(max(0.0, free) / _gr - 1e-9)) * _gr

    gre_block_dur = dur_exc + pre_dur + delay_TE + dur_ro + pre_dur
    inv_duration = pp.calc_duration(rf_inv)
    delay_TI_start = (
        np.ceil((TI_start - inv_duration / 2 - dur_exc / 2) / _gr) * _gr
    )
    assert delay_TI_start > 0, "TI_start too short"
    delay_TI_inter = np.ceil(TI_inter / _gr) * _gr

    # ---- Trajectory -----------------------------------------------------
    def traj(n):
        """n = index of non-dummy readout -> (angle [rad], kz index)"""
        if kz_order == "sequential":
            return np.deg2rad((n // N_z) * GA_DEG), n % N_z
        return np.deg2rad(n * GA_DEG), int(np.floor(((n * GR) % 1.0) * N_z))

    rf_phase = 0.0
    rf_inc = 180.0
    spoke_counter = 0
    trajectory = []  # (angle, kz_index) of every ADC readout, in acquisition order

    for seg in range(N_spokes // R):
        seq.add_block(rf_inv)
        seq.add_block(pp.make_delay(delay_TI_start))

        n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
        for s in range(n_this_seg):
            is_dummy = (seg == 0) and (s < N_dummy)

            rf.phase_offset = rf_phase / 180 * np.pi
            adc.phase_offset = rf_phase / 180 * np.pi
            rf_phase = (rf_phase + rf_inc) % 360

            if is_dummy:
                angle, iz = 0.0, kz_idx_max
            else:
                angle, iz = traj(spoke_counter)
                spoke_counter += 1
                trajectory.append((angle, iz))

            kx, ky = np.cos(angle), np.sin(angle)
            gz_pre, gz_rew = get_gz_pre_rew(iz)

            gx_pre = pp.make_trapezoid(
                channel="x", area=gx_pre_base.area * kx, duration=pre_dur, system=system
            )
            gy_pre = pp.make_trapezoid(
                channel="y", area=gx_pre_base.area * ky, duration=pre_dur, system=system
            )
            gx_rew = pp.make_trapezoid(
                channel="x", area=gx_pre_base.area * kx, duration=pre_dur, system=system
            )
            gy_rew = pp.make_trapezoid(
                channel="y", area=gx_pre_base.area * ky, duration=pre_dur, system=system
            )

            seq.add_block(rf, gz)
            seq.add_block(gx_pre, gy_pre, gz_pre)
            if delay_TE > 0:
                seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(
                channel="x",
                amplitude=gx_base.amplitude * kx,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )
            gy = pp.make_trapezoid(
                channel="y",
                amplitude=gx_base.amplitude * ky,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )

            if is_dummy:
                seq.add_block(gx, gy)
            else:
                seq.add_block(gx, gy, adc)

            seq.add_block(gx_rew, gy_rew, gz_rew)

            if delay_TI_inter > 0 and s < n_this_seg - 1:
                seq.add_block(pp.make_delay(delay_TI_inter))

        seq.add_block(pp.make_delay(TD))

    ok, err = seq.check_timing()
    if ok:
        n_seg = N_spokes // R
        n_total = n_seg * N_readout
        n_full = int(np.ceil(np.pi / 2 * N_x)) * N_z  # radial Nyquist spokes x partitions
        est = n_seg * (inv_duration + delay_TI_start + N_readout * gre_block_dur + TD)
        print("3-D TrueFISP timing check passed")
        print(f"  GRE block dur (TR) : {gre_block_dur * 1000:.2f} ms")
        print(f"  Readout train      : {N_readout * gre_block_dur * 1000:.0f} ms")
        print(f"  Total readouts     : {n_total}  (fully sampled ~{n_full}, "
              f"R_eff ~ {n_full / max(n_total, 1):.1f})")
        print(f"  Scan time est      : {est / 60:.1f} min")
    else:
        print(f"Timing FAILED: {err}")

    seq.set_definition("FOV", [FOV, FOV, FOV_z])
    seq.set_definition("Name", "IR_TrueFISP_bssfp_3D.seq")
    if write_seq:
        seq.write("IR_TrueFISP_bssfp_3D.seq")
    if plot:
        seq.plot(time_range=(0.2, 0.225))

    if return_traj:
        return seq, gre_block_dur, TI_start, np.array(trajectory)
    return seq, gre_block_dur, TI_start

In [ ]:
device = torch.device("cuda")
FOV: float = 250e-3
N_x: int = 128
FOV_z: float = 80e-3
N_channels = 8
N_z = 16           
FOV_z = 80e-3       
N_z_sim = 2 * N_z   
N_z: int = 16
alpha: float = 60.0
slab_thickness: float = 64e-3
TRF: float = 0.8e-3
TBW: float = 2.0
BW: float = 890.0
pre_dur: float = 0.6e-3
TR: float = 3.57e-3
N_readout: int = 1800
N_spokes: int = 1
R=1
N_dummy: int = 0
TI_start: float = 20e-3
TI_inter: float = 0.0
TD: float = 2000e-3
inv_dur: float = 2e-3

In [ ]:
seq, gre_block_dur, TI_start, traj = truefisp_seq_3d(
    FOV=FOV,
    N_x=N_x,
    FOV_z=FOV_z,
    N_z=N_z,
    slab_thickness=slab_thickness,
    alpha=alpha,
    TRF=TRF,
    TBW=TBW,
    BW=BW,
    TR=TR,
    pre_dur=pre_dur,
    inv_dur=inv_dur,
    N_dummy=N_dummy,
    write_seq=True,
    plot=True,
    return_traj=True,
)

print(traj.shape)   # (n_readouts, 2): column 0 = angle [rad], column 1 = kz index

In [ ]:
N_x = 128
n = N_x
# ---- phantom (BrainWeb) ----------------------------------------------------
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(n, n, n),
    device=device,
    permute_order=(2, 1, 0),
)

dz = float(phantom.size[2]) / phantom.PD.shape[2]     # voxel size in z [m]
n_crop = int(round(FOV_z / dz))                       # voxels covering FOV_z
z0 = (phantom.PD.shape[2] - n_crop) // 2              # centred on isocentre
phantom = phantom.slices(list(range(z0, z0 + n_crop)))
print(phantom.PD.shape, phantom.size)
print(type(phantom).__name__, "size =", phantom.size)

# ---------------------------------------------------------------- recon coil sensitivities
# Recon grid: N_x x N_x x N_z. z is scaled by FOV_z / FOV (assumes fov_scaling is
# expressed relative to the in-plane FOV, as in the sim call below).
coil_sensitivities_recon = cyclic_head_coil(
    (N_channels, N_x, N_x, N_z),
    fov_scaling=(1, 1, FOV_z / FOV),
).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2, 3))  # (N_channels, N_x, N_x, N_z)
# all 3 spatial axes, not just (1, 2): the -k NUFFT inverts the image through the
# origin (x, y AND z), so the coil must be flipped in z too for the sense to line up.
# (no [..., 0] any more: the z dimension is kept)
 
# ---------------------------------------------------------------- simulation phantom
# The phantom must be a 3-D voxel grid covering the slab, e.g. resampled to
# (N_x, N_x, N_z_sim) before this point (phantom.interpolate(N_x, N_x, N_z_sim)).
coil_sensitivities_sim = cyclic_head_coil(
    (N_channels, *phantom.PD.shape),          # (N_channels, 128, 128, 128)
    fov_scaling=(
        phantom.size[0] / FOV,
        phantom.size[1] / FOV,
        phantom.size[2] / FOV,
    ),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim.to(device)
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)
 
data = phantom.build()

In [ ]:
# ---------------------------------------------------------------- simulate
mr0_seq = mr0.Sequence.import_file("IR_TrueFISP_bssfp_3D.seq")
torch.manual_seed(0)
signal = mr0.isochromat_sim(
    mr0_seq, data.cuda(), spin_count=64, spin_dist="rand", print_progress=True,
)
signal = signal.permute((1, 0)).contiguous()   # (N_channels, N_samples)
print(f"simulated signal: {tuple(signal.shape)}")

In [ ]:
seq = pp.Sequence()
seq.read("IR_TrueFISP_bssfp_3D.seq")
data = signal.clone()
data = data.contiguous()
kspace = seq.calculate_kspace()
k_phys = kspace[0]
k_phys = k_phys[:3, :]          # (3, S) in cycles/m

# Per-axis normalisation to cycles/voxel on the (N_x, N_x, N_z) recon grid.
# A single global max-radius scale (as in the cubic 3-D Kooshball notebook) is
# only valid when FOV_z == FOV: here the in-plane extent (254 1/m) dominates the
# kz extent (100 1/m), so kz would be squashed to ~32% of its correct scale.
k_traj_adc = np.stack([
    k_phys[0] * FOV / N_x,
    k_phys[1] * FOV / N_x,
    k_phys[2] * FOV_z / N_z,
])
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)
print("k traj:", tuple(k_traj_adc.shape),
      " extents (cyc/vox):", [f"{k_traj_adc[i].abs().max().item():.3f}" for i in range(3)])


In [ ]:
class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities

        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class NonuniformFourierTransformOperator(LinearOperator):
    'F: image (nX,nY,nZ) -> multi-coil k-space (nC, nK). k: (3, nK) in cycles/voxel.'

    def __init__(self, k, input_shape, device=None):
        nD, nK = k.shape                      # (3, nK)
        nC, nX, nY, nZ = input_shape
        self.n_modes = (nX, nY, nZ)
        self.k = (k).unsqueeze(-1)           # (3, nK, 1), sign flipped as in the Kooshball notebooks
        self.forward_shape = (nC, nX, nY, nZ, 1)
        self.adjoint_shape = (nC, nK)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)
        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

class GradientOperator(LinearOperator):

    def __init__(
        self,
        dim: int,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        self.dim = dim
        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = x - torch.roll(x, 1, self.dim)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = x - torch.roll(x, -1, self.dim)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class IdentityOperator(LinearOperator):

    def __init__(
        self,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):

        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        return x


In [ ]:
# ---------------------------------------------------------------- view-sharing parameters
# 3D needs far more spokes per frame than 2D: each frame must cover (angle, kz) well, and a
# window of 55 spokes would put only ~3 spokes on each of the N_z partitions.
N_center = 21       # spokes at the k-space centre (Fibonacci; >= N_z so most kz planes are hit)
N_max = 144         # outermost Fibonacci ring; also the frame time window
frame_advance = 8   # filter advance per frame [TR]
frame_stride = 4  
use_dcf = False     # iterative recon models the exact sampling
 
FIB = [8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987]
 
 
def _next_fib(x):
    return next(f for f in FIB if f >= x)
 
 
assert k_traj_adc.shape[0] == 3, "k_traj_adc must be (3, S): kx, ky, kz"
n_samples_total = k_traj_adc.shape[1]
assert n_samples_total % N_x == 0, "trajectory is not a whole number of spokes"
n_spokes_total = n_samples_total // N_x
assert n_spokes_total > N_max, "need more spokes than N_max for view-sharing"
 
# radius of each radial sample (0 at centre -> 1 at the edge) and its ring count.
# The butterfly is kept in-plane (radial) only: since the sequence uses golden-angle in-plane
# and golden-ratio kz ordering, any Fibonacci run of consecutive spokes is also well
# distributed in kz, so no kz-dependent filter is needed.
r = np.linspace(0.0, 1.0, N_x)
ring = np.array([_next_fib(max(N_center, int(np.ceil(N_max * ri)))) for ri in r])
dcf = np.where(r > 0, 1.0 / r, 1.0 / r[1])
 
# butterfly offsets: for sample i, the ring[i] projections centred on the frame
djs, is_ = [], []
for i in range(N_x):
    nproj = int(ring[i])
    lo = -(nproj // 2)
    djs.extend(range(lo, lo + nproj))
    is_.extend([i] * nproj)
dj = np.asarray(djs, dtype=np.int64)   # (K,) relative projection offset
ii = np.asarray(is_, dtype=np.int64)   # (K,) radial sample index
K = dj.size
w = dcf[ii] if use_dcf else np.ones(K)
 
# valid frames: the outermost window [n - N_max//2, n + N_max//2] must fit
n0 = N_max // 2
n_frames = (n_spokes_total - N_max - 1) // frame_advance + 1
centers_all = n0 + frame_advance * np.arange(n_frames)
 
# ---------------------------------------------------------------- which frames to reconstruct
# Frames are independent, so only reconstruct what is needed:
#   - the LAST frame (steady state -> brain mask / reference phase)
#   - enough frames across the recovery to find the null and to fit T1/T2 (3 unknowns)
rec_idx = np.unique(np.r_[np.arange(0, n_frames, frame_stride), n_frames - 1])
n_rec = rec_idx.size
centers = centers_all[rec_idx]
print(f"reconstructing {n_rec} of {n_frames} frames (stride {frame_stride})")
 
idx = (centers[:, None] + dj[None, :]) * N_x + ii[None, :]   # (n_rec, K)

In [ ]:
# ---------------------------------------------------------------- gather (CPU)
sig_cpu = signal.cpu().numpy()       # (C, S) complex64
k_cpu = k_traj_adc.cpu().numpy()     # (3, S) float32
flat = idx.reshape(-1)               # (n_rec * K,)
k_traj_adc_reshaped = (
    torch.from_numpy(k_cpu[:, flat].reshape(3, n_rec, K).transpose(0, 2, 1).copy())
    .float()
    .to(k_traj_adc.device)
)  # (3, K, n_rec)
data_reshaped = torch.from_numpy(
    (
        sig_cpu[:, flat].reshape(N_channels, n_rec, K).transpose(0, 2, 1).copy()
        * w[None, :, None]
    ).astype(np.complex64)
).to(signal.device)  # (C, K, n_rec)

In [ ]:
# ---------------------------------------------------------------- CG-SENSE (frame by frame)
vol_shape = (N_x, N_x, N_z)
C = ChannelOperator(coil_sensitivities_recon, vol_shape)   # same for every frame
 
x_frames = []
for f in range(n_rec):
    k_f = k_traj_adc_reshaped[:, :, f].contiguous()        # (3, K)
    d_f = data_reshaped[:, :, f].contiguous()              # (C, K)
 
    F = NonuniformFourierTransformOperator(k_f, (N_channels, *vol_shape))
    A = F @ C
    AHd = A.H @ d_f.view(torch.float32).ravel()
    Q = A.H @ A
    x0 = torch.zeros(int(np.prod(vol_shape)), dtype=torch.complex64).view(torch.float32)
    x_f = conjugate_gradient(Q, AHd, x0, maxiter=30)
    x_frames.append(x_f.view(torch.complex64).reshape(vol_shape).cpu())
    if f % 10 == 0:
        print(f"  frame {f + 1}/{n_rec}")
 
x_recon_image = torch.stack(x_frames, dim=-1)              # (N_x, N_x, N_z, n_rec)
x_recon_image = x_recon_image.flip((0, 1, 2))   # undo the -k NUFFT origin-inversion so the axes line up with the phantom
print(f"reconstructed {n_rec} frames, shape {tuple(x_recon_image.shape)}")

In [ ]:
# ---------------------------------------------------------------- analysis
ximg = x_recon_image.numpy()         # (N_x, N_x, N_z, Nf) complex, Nf = n_rec
Nf = ximg.shape[3]
t_frames = centers * gre_block_dur * 1e3  # ms after first readout
phase = np.angle(ximg[..., -1])           # steady-state phase, (N_x, N_x, N_z)
signed = np.real(ximg * np.exp(-1j * phase)[..., None])
mag = np.abs(ximg)
print(
    f"frames: {Nf}, spacing {frame_stride * frame_advance * TR * 1e3:.0f} ms; "
    f"span {t_frames[0]:.0f}-{t_frames[-1]:.0f} ms"
)

ref = mag[..., -1]                   # steady-state magnitude volume
brain = ref > 0.1 * ref.max()        # (N_x, N_x, N_z)

# Ground-truth T1 map (phantom, in seconds) lives on the sim grid
# (N_x, N_x, ~41); resample it to the recon grid (N_x, N_x, N_z).
import torch.nn.functional as tF
T1m_t = phantom.T1.detach().cpu()[None, None]
T1m_rec = tF.interpolate(T1m_t, size=(N_x, N_x, N_z), mode="trilinear")[0, 0].numpy()

tissues = [
    ("WM", brain & (T1m_rec < 1.0), "tab:blue"),
    ("GM", brain & (T1m_rec >= 1.0) & (T1m_rec <= 2.0), "tab:orange"),
    ("CSF", brain & (T1m_rec > 2.0), "tab:green"),
]

# Fit mask: tissue CORE only (>= core_dist voxels from every interface).
# Partial-volume voxels follow a mixture of tissue curves, not the single
# exponential of the IR model, so they are excluded from the T1/T2/PD fit.
from scipy.ndimage import distance_transform_edt
core_dist = 3
fit_mask = np.zeros((N_x, N_x, N_z), dtype=bool)
for _name, _tm, _color in tissues:
    fit_mask |= _tm & (distance_transform_edt(_tm) >= core_dist)
print(f"fit mask: {fit_mask.sum()} tissue-core voxels (>= {core_dist} from every interface)")


In [ ]:
vmax = 1.5 * float(np.percentile(mag, 95))
zc = N_z // 2   # slice shown in the series
 
fnull = int(np.argmin(mag.reshape(-1, Nf)[brain.reshape(-1)].mean(0)))  # darkest reconstructed frame
 
# 11 frames spread over 0-1200 ms (nearest reconstructed frame) + the null frame
t_target = np.linspace(t_frames[0], 1200.0, 11)
sel_pos = np.abs(t_frames[None, :] - t_target[:, None]).argmin(axis=1)
frame_idx = np.sort(np.unique(np.concatenate([sel_pos, [fnull]])))
 
nrows, ncols = 3, (len(frame_idx) + 2) // 3
plt.figure(figsize=(ncols * 1.5, nrows * 1.5))
plt.suptitle(f"IR magnitude series, slice {zc}/{N_z} (grayscale) - darkest frame = null")
for j, i in enumerate(frame_idx):
    ax = plt.subplot(nrows, ncols, j + 1)
    is_null = i == fnull
    plt.imshow(mag[:, :, zc, i].T, cmap="gray", vmin=0, vmax=vmax)
    plt.title(
        f"{t_frames[i]:.0f} ms",
        fontsize=9,
        color="red" if is_null else "k",
        fontweight="bold" if is_null else "normal",
    )
    for s_ in ax.spines.values():
        s_.set_edgecolor("red" if is_null else "0.8")
        s_.set_linewidth(2.5 if is_null else 0.5)
    plt.axis("off")
for j in range(len(frame_idx), nrows * ncols):
    plt.subplot(nrows, ncols, j + 1)
    plt.axis("off")
plt.show()

In [ ]:
n_use = 987
s_idx = np.arange((n_spokes_total - n_use) * N_x, n_spokes_total * N_x)
k_t = k_traj_adc[:, s_idx].contiguous()
d_t = signal[:, s_idx].contiguous()

F = NonuniformFourierTransformOperator(k_t, (N_channels, N_x, N_x, N_z))
A = F @ ChannelOperator(coil_sensitivities_recon, (N_x, N_x, N_z))
b = A.H @ d_t.view(torch.float32).ravel()
x = conjugate_gradient(A.H @ A, b, torch.zeros_like(b), maxiter=30)
img = x.view(torch.complex64).reshape(N_x, N_x, N_z).abs().cpu().numpy()
# plt.imshow(img[:, :, N_z // 2].T, cmap="gray")

In [ ]:
plt.figure(figsize=(10, 3))
for j, z in enumerate([4, 8, 12]):
    plt.subplot(1, 3, j + 1)
    plt.imshow(img[:, :, z].T, cmap="gray")
    plt.title(f"z = {z}")
    plt.axis("off")
plt.show()

In [ ]:
# ---------------------------------------------------------------- DeepADMM (3-D)
class DeepADMM3D(nn.Module):
    """3-D port of the 2-D notebook's DeepADMM.

    ADMM solve of  min_x  1/2 ||A x - d||^2   s.t.  x = regularizer(x),
    with a learned complex 3-D ResNet regularizer.  The kernel is (3,3,3),
    i.e. 3-D convolutions, so - unlike the 2-D (3,3) kernel - the z
    dimension is not killed.  rho and the ResNet weights are trained
    self-supervised on k-space source/target splits (next cell).
    """

    def __init__(self, shape, rho_init=0.1, maxiter_admm=10, maxiter_cg=5,
                 features=32, num_of_resblocks=10, checkpoints=True):
        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(
            shape,
            kernel_size=(3, 3, 3),
            features=features,
            num_of_resblocks=num_of_resblocks,
            Checkpoints=checkpoints,
        )
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg

    def forward(self, Q, b, x, disable_progressbar=True):
        z = torch.zeros(self.I.shape[0], dtype=x.dtype)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)

        for iteration in range(self.maxiter_admm):
            x = conjugate_gradient(
                Q + self.rho * self.I,
                b=b + self.rho * (z - u),
                x=x,
                maxiter=self.maxiter_cg,
                disable_progressbar=disable_progressbar,
            )
            z = self.regularizer(x + u)
            u = u + x - z

        return x

In [ ]:
# ---------------------------------------------------------------- train (3-D)
# Self-supervised k-space split training, port of the 2-D notebook: per frame,
# reconstruct from HALF of the butterfly's k-space points with the ADMM model
# and penalise the disagreement with the held-out half.
shape3d = (N_x, N_x, N_z)
torch.manual_seed(0)
model = DeepADMM3D(
    shape3d, rho_init=0.1, maxiter_admm=5, maxiter_cg=4,
    features=32, num_of_resblocks=8, checkpoints=True,
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Kb = k_traj_adc_reshaped.shape[1]
C1 = ChannelOperator(coil_sensitivities_recon, shape3d)
sel_train = np.arange(n_rec)      # train on all reconstructed frames
num_iters = 30

model.train()
losses = []
for it in tqdm(range(num_iters), desc="Self-supervised splits (3D)"):
    np.random.seed(it)
    f = sel_train[np.random.randint(sel_train.size)]

    indices = np.arange(Kb)
    np.random.shuffle(indices)
    index_src, index_tar = np.array_split(indices, 2)

    k_src = k_traj_adc_reshaped[:, index_src, f]
    k_tar = k_traj_adc_reshaped[:, index_tar, f]
    d_src = data_reshaped[:, index_src, f].contiguous().view(torch.float32).ravel()
    d_tar = data_reshaped[:, index_tar, f].contiguous().view(torch.float32).ravel()

    F_src = NonuniformFourierTransformOperator(k_src, (N_channels, *shape3d))
    F_tar = NonuniformFourierTransformOperator(k_tar, (N_channels, *shape3d))

    Q_src = C1.H @ F_src.H @ F_src @ C1
    b_src = C1.H @ F_src.H @ d_src
    x_out = model(Q_src, b_src, b_src, disable_progressbar=True)
    k_data_pred = F_tar @ C1 @ x_out

    loss = torch.linalg.norm(k_data_pred - d_tar, ord=1) / torch.linalg.norm(d_tar, ord=1)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    tqdm.write(f"\rloss: {loss.item():.4f}")

plt.figure()
plt.plot(losses, marker="o")
plt.xlabel("split"); plt.ylabel("self-supervised loss")
plt.title("DeepADMM training loss (3D)")
plt.show()
torch.save(model.state_dict(), "deepadmm3d.pt")

In [ ]:
# ---------------------------------------------------------------- DeepADMM inference (3-D)
model.eval()
# no autograd at eval time -> more CG/ADMM iterations are cheap
model.maxiter_admm = 6
model.maxiter_cg = 6
recon3d = []
for f in tqdm(range(n_rec), desc="DeepADMM inference"):
    k_f = k_traj_adc_reshaped[:, :, f].contiguous()
    d_f = data_reshaped[:, :, f].contiguous()
    F_f = NonuniformFourierTransformOperator(k_f, (N_channels, *shape3d))
    C_f = ChannelOperator(coil_sensitivities_recon, shape3d)
    A_f = F_f @ C_f
    b_f = A_f.H @ d_f.view(torch.float32).ravel()
    with torch.no_grad():
        x_f = model(A_f.H @ A_f, b_f, b_f)
    recon3d.append(x_f.view(torch.complex64).reshape(shape3d).cpu())

x_deepadmm_image = torch.stack(recon3d, dim=-1)   # (N_x, N_x, N_z, n_rec)
x_deepadmm_image = x_deepadmm_image.flip((0, 1, 2))   # undo the -k NUFFT origin-inversion so the axes line up with the phantom
print(f"DeepADMM frames: {tuple(x_deepadmm_image.shape)}")
torch.save(x_deepadmm_image.cpu(), "deepadmm3d_frames.pt")

In [ ]:
# ---------------------------------------------------------------- DeepADMM series (3-D)
dimg = x_deepadmm_image.numpy()               # (N_x, N_x, N_z, Nf)
Nf_da = dimg.shape[3]
phase_da = np.angle(dimg[..., -1])            # steady-state phase
signed_da = np.real(dimg * np.exp(-1j * phase_da)[..., None])
mag_da = np.abs(dimg)

ref_da = mag_da[..., -1]
brain_da = ref_da > 0.1 * ref_da.max()

fnull_da = int(np.argmin(mag_da.reshape(-1, Nf_da)[brain_da.reshape(-1)].mean(0)))

# 11 frames spread over 0-1200 ms (nearest reconstructed frame) + the null
t_target = np.linspace(t_frames[0], 1200.0, 11)
sel_pos = np.abs(t_frames[None, :] - t_target[:, None]).argmin(axis=1)
frame_idx_da = np.sort(np.unique(np.concatenate([sel_pos, [fnull_da]])))

vmax_da = 1.5 * float(np.percentile(mag_da, 95))
zc = N_z // 2
nrows, ncols = 3, (len(frame_idx_da) + 2) // 3
plt.figure(figsize=(ncols * 1.5, nrows * 1.5))
plt.suptitle(f"DeepADMM magnitude series, slice {zc}/{N_z} (grayscale) - darkest frame = null")
for j, i in enumerate(frame_idx_da):
    ax = plt.subplot(nrows, ncols, j + 1)
    is_null = i == fnull_da
    plt.imshow(mag_da[:, :, zc, i].T, cmap="gray", vmin=0, vmax=vmax_da)
    plt.title(
        f"{t_frames[i]:.0f} ms",
        fontsize=9,
        color="red" if is_null else "k",
        fontweight="bold" if is_null else "normal",
    )
    for s_ in ax.spines.values():
        s_.set_edgecolor("red" if is_null else "0.8")
        s_.set_linewidth(2.5 if is_null else 0.5)
    plt.axis("off")
for j in range(len(frame_idx_da), nrows * ncols):
    plt.subplot(nrows, ncols, j + 1)
    plt.axis("off")
plt.show()

In [ ]:
# ---------------------------------------------------------------- T1/T2/PD fit (3-D)
# Signal model (TrueFISP IR, same Eq. 1 as the 2-D notebook):
#     S(t) = S_StSt - (S0 + S_StSt) * exp(-t / T1star)
# with t the time after the inversion pulse, S0 = -S(0) and S_StSt = S(infinity).
# Fit per voxel with separable NLLS: grid over T1star, closed-form linear solve
# for (S_StSt, S0) at each grid point -> equivalent to a full 3-parameter NLLS
# up to grid resolution.  Uses the PHASE-CORRECTED SIGNED DeepADMM signal so
# there is no magnitude sign ambiguity around the null.

time_ms = TI_start * 1e3 + t_frames          # ms after inversion
time_s = torch.tensor(time_ms / 1e3, dtype=torch.float64, device=device)
y = torch.tensor(signed_da, dtype=torch.float64, device=device)   # (..., Nf)
V, Nf = N_x * N_x * N_z, y.shape[-1]
y_flat = y.reshape(V, Nf)

mask = torch.tensor(fit_mask.reshape(-1), device=device)
y_in = y_flat[mask]
Vm = y_in.shape[0]
print(f"fitting {Vm} tissue-core voxels x {Nf} frames")

T1star_grid = torch.logspace(np.log10(0.02), np.log10(5.0), 400,
                             device=device, dtype=torch.float64)
ones = torch.ones(Nf, dtype=torch.float64, device=device)

best_sse = torch.full((Vm,), float("inf"), dtype=torch.float64, device=device)
best_c1 = torch.zeros(Vm, dtype=torch.float64, device=device)
best_c2 = torch.zeros(Vm, dtype=torch.float64, device=device)
best_t1star = torch.zeros(Vm, dtype=torch.float64, device=device)

for t1star_candidate in tqdm(T1star_grid, desc="T1star grid"):
    f = torch.exp(-time_s / t1star_candidate)
    S_ff, S_f1, S_11 = (f * f).sum(), f.sum(), float(Nf)
    S_yf, S_y1 = y_in @ f, y_in @ ones
    det = S_11 * S_ff - S_f1 * S_f1
    c1 = (S_y1 * S_ff - S_yf * S_f1) / det
    c2 = (S_yf * S_11 - S_y1 * S_f1) / det
    sse = ((y_in - c1[:, None] * ones - c2[:, None] * f) ** 2).sum(dim=1)
    better = sse < best_sse
    best_sse[better], best_c1[better], best_c2[better], best_t1star[better] = (
        sse[better], c1[better], c2[better], t1star_candidate)

S_StSt = torch.zeros(V, dtype=torch.float64, device=device)
S0 = torch.zeros(V, dtype=torch.float64, device=device)
T1star = torch.zeros(V, dtype=torch.float64, device=device)
S_StSt[mask] = best_c1
S0[mask] = -best_c2 - best_c1
T1star[mask] = best_t1star

In [ ]:
# ---------------------------------------------------------------- parameter maps (3-D)
alpha_rad = np.deg2rad(alpha)
cos_a2, sin_a2 = np.cos(alpha_rad / 2), np.sin(alpha_rad / 2)

S_StSt_np = S_StSt.cpu().numpy().reshape(N_x, N_x, N_z)
S0_np = S0.cpu().numpy().reshape(N_x, N_x, N_z)
T1star_np = T1star.cpu().numpy().reshape(N_x, N_x, N_z)

T1_map = np.zeros_like(T1star_np)
T2_map = np.zeros_like(T1star_np)
PD_map = np.zeros_like(T1star_np)
_ok = fit_mask & (np.abs(S_StSt_np) > 1e-12) & (np.abs(S0_np) > 1e-12)
T1_map[_ok] = T1star_np[_ok] * (S0_np[_ok] / S_StSt_np[_ok]) * cos_a2
T2_map[_ok] = T1star_np[_ok] * sin_a2**2 / (1 - (S_StSt_np[_ok] / S0_np[_ok]) * cos_a2)
PD_map[_ok] = S0_np[_ok] / sin_a2
T1_map[~fit_mask] = 0
T2_map[~fit_mask] = 0
PD_map[~fit_mask] = 0
print(f"T1 range: {np.percentile(T1_map[fit_mask],1)*1e3:.0f} - "
      f"{np.percentile(T1_map[fit_mask],99)*1e3:.0f} ms")

In [ ]:
# ---------------------------------------------------------------- maps (3-D)
zc = N_z // 2
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

im0 = axes[0].imshow(PD_map[:, :, zc].T, cmap="gray",
                     vmin=0, vmax=np.percentile(PD_map[fit_mask], 99))
axes[0].set_title("Proton Density"); axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(T1_map[:, :, zc].T * 1e3, cmap="hot", vmin=0, vmax=4500)
axes[1].set_title("T_1 [ms]"); axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(T2_map[:, :, zc].T * 1e3, cmap="hot", vmin=0, vmax=500)
axes[2].set_title("T_2 [ms]"); axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle(f"Quantitative maps, slice {zc}/{N_z}")
plt.tight_layout()
plt.show()

# tissue averages (median) vs the phantom ground truth
gt = {"WM": (900, 70, 0.7), "GM": (1300, 80, 0.8), "CSF": (4000, 500, 1.0)}
print(f"{'tissue':6s} {'n vox':>8s} {'T1 [ms]':>9s} {'T2 [ms]':>9s} {'PD':>6s}   GT (T1, T2, PD)")
for _name, _tm, _color in tissues:
    _m = fit_mask & _tm
    if _m.sum() == 0:
        continue
    print(f"{_name:6s} {_m.sum():8d} "
          f"{np.median(T1_map[_m])*1e3:9.0f} "
          f"{np.median(T2_map[_m])*1e3:9.0f} "
          f"{np.median(PD_map[_m]):6.2f}   "
          f"{gt[_name][0]:4.0f}  {gt[_name][1]:4.0f}  {gt[_name][2]:.1f}")